# 🧪 Sesión 36 · Arquitectura Single Page Application (SPA) Ligera

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo navegas entre vistas sin recargar la página?

---

## 🖥️ Así se verá el resultado final

```text
http://localhost:8000/#/libros
  -> vista de lista de libros

http://localhost:8000/#/libros/1
  -> vista de detalle del libro 1

http://localhost:8000/#/no-existe
  -> vista 404

Al cambiar el hash no se recarga el documento:
solo se reemplaza el contenido de <div id="app">.
```

_El hash no viaja al servidor: el navegador lo entrega a JavaScript y el ruteador decide._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un ruteador por hash.
- Renderizar una vista según la URL.
- Verificar que la navegación no recarga el documento.

## ✅ Criterios de aceptación

El verificador comprobará estos criterios sobre los archivos que produzcas:

- [ ] El ruteador escucha el hash y renderiza vistas
- [ ] La página contiene el contenedor y los enlaces

## 🧩 El reto

Implementa una SPA ligera con rutas por hash para listar y ver el detalle de libros.

**Requisitos del contrato**

- Escucha el evento hashchange.
- La ruta #/libros muestra la lista.
- La ruta #/libros/{id} muestra el detalle.
- Una ruta desconocida muestra 404.
- La vista se renderiza dentro de #app sin recargar la página.

> **Pista conceptual:** El ruteador es una función que lee `location.hash`, elige una vista y la pinta. Si usas enlaces con href="#/...", el navegador no recarga.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion36.md` con la captura de las tres rutas.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `index.html`, `router.js`


### 📄 `index.html`

Página única de la SPA.

In [2]:
%%writefile index.html
<!DOCTYPE html>
<html lang="es">
<head><meta charset="UTF-8"><title>Biblioteca SPA</title></head>
<body>
  <nav>
    <a href="#/libros">Libros</a>
  </nav>
  <div id="app"></div>
  <script src="router.js"></script>
</body>
</html>


Writing index.html


### 📄 `router.js`

Ruteador y vistas.

In [3]:
%%writefile router.js
// Ruteador por hash y vistas para la SPA de biblioteca.
const API = "/api/libros"; // Ajusta la URL a tu backend
const app = document.getElementById("app");

const rutas = {
  "#/libros": vistaLista,
  "#/libros/nuevo": vistaFormulario,
};

function renderRoute() {
  const hash = location.hash || "#/libros";
  // Rutas con parámetro: #/libros/editar/3
  const editar = hash.match(/^#\/libros\/editar\/(\w+)$/);
  if (editar) return vistaFormulario(editar[1]);

  const vista = rutas[hash];
  if (vista) return vista();
  app.innerHTML = "<h2>404</h2><p>Ruta no encontrada.</p>";
}

async function vistaLista() {
  app.innerHTML = "<p>Cargando...</p>";
  try {
    const res = await fetch(API);
    if (!res.ok) throw new Error("HTTP " + res.status);
    const libros = await res.json();
    app.innerHTML = `
      <h2>Libros</h2>
      <a href="#/libros/nuevo">+ Nuevo libro</a>
      <ul>
        ${libros.map(l => `
          <li>
            ${l.titulo} — ${l.autor}
            <a href="#/libros/editar/${l.id}">Editar</a>
            <button data-id="${l.id}" class="eliminar">Eliminar</button>
          </li>`).join("")}
      </ul>`;
    app.querySelectorAll(".eliminar").forEach(btn =>
      btn.addEventListener("click", async () => {
        await fetch(`${API}/${btn.dataset.id}`, { method: "DELETE" });
        renderRoute();
      })
    );
  } catch (e) {
    app.innerHTML = `<p>Error al cargar libros: ${e.message}</p>`;
  }
}

async function vistaFormulario(id) {
  const editando = typeof id === "string";
  let libro = { titulo: "", autor: "" };
  if (editando) {
    const res = await fetch(`${API}/${id}`);
    libro = await res.json();
  }
  app.innerHTML = `
    <h2>${editando ? "Editar" : "Nuevo"} libro</h2>
    <input id="titulo" placeholder="Título" value="${libro.titulo}">
    <input id="autor" placeholder="Autor" value="${libro.autor}">
    <button id="guardar">Guardar</button>
    <a href="#/libros">Cancelar</a>`;
  document.getElementById("guardar").addEventListener("click", async () => {
    const datos = {
      titulo: document.getElementById("titulo").value,
      autor: document.getElementById("autor").value,
    };
    await fetch(editando ? `${API}/${id}` : API, {
      method: editando ? "PUT" : "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify(datos),
    });
    location.hash = "#/libros";
  });
}

window.addEventListener("hashchange", renderRoute);
window.addEventListener("DOMContentLoaded", renderRoute);


Writing router.js


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("router.js", regex=["hashchange", "location\\.hash", "#/libros", "renderRoute|function\\s+render", "fetch\\(", "app"], descripcion="El ruteador escucha el hash y renderiza vistas"))
resultados.append(revisar_archivo("index.html", exigir=["id=\"app\"", "router.js", "href=\"#/libros\""], descripcion="La página contiene el contenedor y los enlaces"))
resumen(resultados)


✅ El ruteador escucha el hash y renderiza vistas
✅ La página contiene el contenedor y los enlaces
------------------------------------------------------------------------------
RESULTADO: 2/2 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion36.md` a `auditorias/auditoria_sesion36.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
